 ### Property Graph を使って Select AI の精度を高める

In [1]:
# 準備
import dotenv, os, subprocess
import select_ai
import oracledb
import pandas as pd

dotenv.load_dotenv()
user = os.getenv("user")
password = os.getenv("password")
dsn=os.getenv("dsn")
admin_password=os.getenv("admin_password")

#### 関数の定義

In [2]:
def execute_sql_script(script, print_stdout=False):
    """ SQLスクリプトを実行する """
    print(f"script={script}")
    cmd = f"""sql "{user}/{password}@{dsn}" @{script}"""
    result = subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True)
    if print_stdout:
        print(result.stdout)

def execute_sql(sql, print_sql=False):
    """ SQL文を実行する、Pandasで結果を表示する """
    if print_sql:
        print(f"sql={sql}")
    with oracledb.connect(user=user, password=password, dsn=dsn) as con:
        with con.cursor() as cur:
            cur.execute(sql)
            columns = [column[0] for column in cur.description]
            df = pd.DataFrame(cur.fetchall(), columns=columns)
            display(df.style.hide(axis="index"))

def execute_query(profile, query):
    """ Select AI を使って自然言語の問い合わせを実行する """
    try:
        profile = select_ai.Profile(profile_name=profile)
        sql = profile.show_sql(query)
        print(f"### profile={profile.profile_name}")
        print(query)
        print("--- SQL ---")
        print(sql)
        print("--- RESULT ---")
        execute_sql(sql)
    except Exception as e:
        print(e)

#### テーブルの作成

In [3]:
execute_sql_script("ddl.sql")
execute_sql_script("dml.sql")

script=ddl.sql


script=dml.sql


In [4]:
# 作成したテーブルの表示
execute_sql("select * from DEMO.RESIDENTS", print_sql=True)
execute_sql("select * from DEMO.RESIDENTS_SUB1", print_sql=True)
execute_sql("select * from DEMO.RESIDENTS_SUB2", print_sql=True)
execute_sql("select * from DEMO.RESIDENTS_SUB3", print_sql=True)

sql=select * from DEMO.RESIDENTS


ID,NAME
1,Michael
2,John
3,Patty


sql=select * from DEMO.RESIDENTS_SUB1


ID,VALUE
1,1000
2,1200
3,1900


sql=select * from DEMO.RESIDENTS_SUB2


ID,VALUE
1,800
2,1500
3,2000


sql=select * from DEMO.RESIDENTS_SUB3


ID,SUB_ID,KIND,VALUE
1,1,食費,200
2,1,その他,600
3,2,食費,500
4,2,その他,1000
5,3,食費,600
6,3,その他,1300


#### SELECT AI の準備

In [5]:
# Database に接続
select_ai.connect(user=user, password=password, dsn=dsn)

In [6]:
# Resource Principal の設定
with oracledb.connect(user="admin", password=admin_password, dsn=dsn) as con:
    with con.cursor() as cur:
        cur.execute(f"call DBMS_CLOUD_ADMIN.ENABLE_RESOURCE_PRINCIPAL(username => '{user}')")

DEMO_PROFILE_PLAIN プロファイルの作成 (リレーショナルテーブルが対象)と SELECT AI 実行

In [7]:
provider = select_ai.OCIGenAIProvider(
    region="us-chicago-1", 
    model="xai.grok-4.3",
    oci_apiformat="GENERIC"
)
profile_attributes = select_ai.ProfileAttributes(
    credential_name="OCI$RESOURCE_PRINCIPAL",
    object_list=[
                    {"owner": "DEMO", "name": "RESIDENTS"},
                    {"owner": "DEMO", "name": "RESIDENTS_SUB1"},
                    {"owner": "DEMO", "name": "RESIDENTS_SUB2"},
                    {"owner": "DEMO", "name": "RESIDENTS_SUB3"}
                 ],
    provider=provider,
    max_tokens=4096,
    comments=True
)
profile = select_ai.Profile(
    profile_name="DEMO_PROFILE_PLAIN",
    attributes=profile_attributes,
    description="AI Profile for Relational tables",
    replace=True,
)

In [8]:
execute_query("DEMO_PROFILE_PLAIN", "住民の名前を全員表示して下さい")

### profile=DEMO_PROFILE_PLAIN
住民の名前を全員表示して下さい
--- SQL ---
SELECT r."NAME" AS "Resident_Name"
FROM "DEMO"."RESIDENTS" r
--- RESULT ---


Resident_Name
Michael
John
Patty


In [9]:
execute_query("DEMO_PROFILE_PLAIN", "各住民の収入はいくらですか")

### profile=DEMO_PROFILE_PLAIN
各住民の収入はいくらですか
--- SQL ---
SELECT "r"."NAME" AS "resident_name",
       "s1"."VALUE" AS "income"
FROM "DEMO"."RESIDENTS" "r"
JOIN "DEMO"."RESIDENTS_SUB1" "s1"
  ON "r"."ID" = "s1"."ID"
--- RESULT ---


resident_name,income
Michael,1000
John,1200
Patty,1900


In [10]:
execute_query("DEMO_PROFILE_PLAIN", "支出が一番多い住民と支出額は？")

### profile=DEMO_PROFILE_PLAIN
支出が一番多い住民と支出額は？
--- SQL ---
SELECT r."NAME" AS "RESIDENT_NAME", SUM(v."VALUE") AS "TOTAL_EXPENDITURE"
FROM "DEMO"."RESIDENTS" r
JOIN (SELECT "ID","VALUE" FROM "DEMO"."RESIDENTS_SUB1" UNION ALL SELECT "ID","VALUE" FROM "DEMO"."RESIDENTS_SUB2" UNION ALL SELECT "ID","VALUE" FROM "DEMO"."RESIDENTS_SUB3") v ON r."ID"=v."ID"
GROUP BY r."NAME"
ORDER BY "TOTAL_EXPENDITURE" DESC FETCH FIRST 1 ROW ONLY
--- RESULT ---


RESIDENT_NAME,TOTAL_EXPENDITURE
Patty,4400


In [11]:
execute_query("DEMO_PROFILE_PLAIN", "支出に占める食費の割合が一番高い住民の名前とパーセンテージを教えて")

### profile=DEMO_PROFILE_PLAIN
支出に占める食費の割合が一番高い住民の名前とパーセンテージを教えて
--- SQL ---
SELECT r."NAME" AS "名前", (s3."VALUE"/s1."VALUE"*100) AS "パーセンテージ"
FROM "DEMO"."RESIDENTS" r
JOIN "DEMO"."RESIDENTS_SUB1" s1 ON r."ID"=s1."ID"
JOIN "DEMO"."RESIDENTS_SUB3" s3 ON r."ID"=s3."ID"
WHERE UPPER(s3."KIND")=UPPER('食費')
ORDER BY "パーセンテージ" DESC
FETCH FIRST 1 ROWS ONLY
--- RESULT ---


名前,パーセンテージ
Patty,26.315789


#### DEMO_PROFILE_GRAPH プロファイルの作成 (Property Graphが対象)と SELECT AI 実行

In [12]:
# SQL Property Graph の作成
execute_sql_script("create_graph.sql")

script=create_graph.sql


In [13]:
# DEMO_PROFILE_GRAPH を作成
provider = select_ai.OCIGenAIProvider(
    region="us-chicago-1", 
    model="xai.grok-4.3",
    oci_apiformat="GENERIC"
)
profile_attributes = select_ai.ProfileAttributes(
    credential_name="OCI$RESOURCE_PRINCIPAL",
    object_list=[
                    {"owner": "DEMO", "name": "RESIDENTS_GRAPH"}
                 ],
    provider=provider,
    max_tokens=4096,
    comments=True
)
profile = select_ai.Profile(
    profile_name="DEMO_PROFILE_GRAPH",
    attributes=profile_attributes,
    description="AI Profile for Graph tables",
    replace=True,
)

In [14]:
execute_query("DEMO_PROFILE_GRAPH", "住民の名前を全員表示して下さい")

### profile=DEMO_PROFILE_GRAPH
住民の名前を全員表示して下さい
--- SQL ---
SELECT resident_name
FROM GRAPH_TABLE("DEMO"."RESIDENTS_GRAPH"
MATCH (v IS "RESIDENT")
COLUMNS (v."NAME" AS resident_name))
--- RESULT ---


RESIDENT_NAME
Michael
John
Patty


In [15]:
execute_query("DEMO_PROFILE_GRAPH", "各住民の収入はいくらですか")

### profile=DEMO_PROFILE_GRAPH
各住民の収入はいくらですか
--- SQL ---
SELECT resident_name, income_amount
FROM GRAPH_TABLE("DEMO"."RESIDENTS_GRAPH"
MATCH (resident IS "RESIDENT") -[has_income IS "HAS_INCOME"]-> (income IS "INCOME")
COLUMNS (resident."NAME" AS resident_name, income."AMOUNT" AS income_amount))
--- RESULT ---


RESIDENT_NAME,INCOME_AMOUNT
Michael,1000
John,1200
Patty,1900


In [16]:
execute_query("DEMO_PROFILE_GRAPH", "支出が一番多い住民と支出額は？")

### profile=DEMO_PROFILE_GRAPH
支出が一番多い住民と支出額は？
--- SQL ---
SELECT resident_name, expense_amount
FROM GRAPH_TABLE("DEMO"."RESIDENTS_GRAPH"
MATCH (resident IS "RESIDENT") -[has_exp IS "HAS_EXPENSE"]-> (expense IS "EXPENSE")
COLUMNS (resident."NAME" AS resident_name, expense."AMOUNT" AS expense_amount))
ORDER BY expense_amount DESC
FETCH FIRST 1 ROW ONLY
--- RESULT ---


RESIDENT_NAME,EXPENSE_AMOUNT
Patty,2000


In [17]:
execute_query("DEMO_PROFILE_GRAPH", "支出に占める食費の割合が一番高い住民の名前とパーセンテージを教えて")

### profile=DEMO_PROFILE_GRAPH
支出に占める食費の割合が一番高い住民の名前とパーセンテージを教えて
--- SQL ---
SELECT name,
       ROUND(food_amount / NULLIF(total_amount, 0) * 100, 2) AS percentage
FROM (
  SELECT r_name AS name,
         SUM(CASE WHEN UPPER(kind) = UPPER('食費') THEN amount ELSE 0 END) AS food_amount,
         SUM(amount) AS total_amount
  FROM GRAPH_TABLE("DEMO"."RESIDENTS_GRAPH"
    MATCH (r IS "RESIDENT") -[e1 IS "HAS_EXPENSE"]-> (exp IS "EXPENSE") -[e2 IS "HAS_DETAIL"]-> (det IS "EXPENSE_DETAIL")
    COLUMNS (r."NAME" AS r_name, det."KIND" AS kind, det."AMOUNT" AS amount)
  )
  GROUP BY r_name
)
ORDER BY percentage DESC NULLS LAST
FETCH FIRST 1 ROW ONLY
--- RESULT ---


NAME,PERCENTAGE
John,33.330000


#### Property Graph の Select AI はどんなプロンプトを使っているか

In [ ]:
profile = select_ai.Profile(profile_name="DEMO_PROFILE_GRAPH")
prompt = profile.show_prompt("支出に占める食費の割合が一番高い住民の名前とパーセンテージを教えて")

import json

instructions = json.loads(prompt) # type:list
for inst in instructions:
    print("########")
    print(inst["content"][0]["text"].replace("\\n", "\n"))

########
# Role and Objective
You are an Oracle SQL/PGQ expert.
SQL/PGQ is an extension of the SQL query language for the property graph data
model that allows users to specify high-level graph patterns matched against
vertices and edges. It integrates with all existing SQL functionalities and is
supported in Oracle Database.
Your role is to help users define and query property graphs using SQL/PGQ.
Always reason step-by-step, produce correct and complete queries, and apply
best practices from Oracle documentation.
# Context and Knowledge Use
Note the following rules about Oracle SQL/PGQ:

- to access graph data, use the GRAPH_TABLE operator, which can be used anywhere
  a table can be used
- property graph comments, when present in the context, describe the graph's
  business meaning and intended use
- use property graph comments as semantic guidance when choosing the graph and
  interpreting labels, properties and relationships
- do not invent labels, properties or relationships that

#### Graph を使わなくてもプロンプトを工夫すれば同じことはできる

In [19]:
execute_query("DEMO_PROFILE_PLAIN", """
relationships:
  # 住民と収入
  - from: { table: DEMO.RESIDENTS_SUB1, columns: [ID] }
    to:   { table: DEMO.RESIDENTS, columns: [ID] }
    cardinality: one_to_one

  # 住民と支出
  - from: { table: DEMO.RESIDENTS_SUB2, columns: [ID] }
    to:   { table: DEMO.RESIDENTS, columns: [ID] }
    cardinality: one_to_one

  # 支出と支出明細
  - from: { table: DEMO.RESIDENTS_SUB3, columns: [SUB_ID] }
    to:   { table: DEMO.RESIDENTS_SUB2, columns: [ID] }
    cardinality: many_to_one

Query: 
支出に占める食費の割合が一番高い住民の名前とパーセンテージを教えて
""")

### profile=DEMO_PROFILE_PLAIN

relationships:
  # 住民と収入
  - from: { table: DEMO.RESIDENTS_SUB1, columns: [ID] }
    to:   { table: DEMO.RESIDENTS, columns: [ID] }
    cardinality: one_to_one

  # 住民と支出
  - from: { table: DEMO.RESIDENTS_SUB2, columns: [ID] }
    to:   { table: DEMO.RESIDENTS, columns: [ID] }
    cardinality: one_to_one

  # 支出と支出明細
  - from: { table: DEMO.RESIDENTS_SUB3, columns: [SUB_ID] }
    to:   { table: DEMO.RESIDENTS_SUB2, columns: [ID] }
    cardinality: many_to_one

Query: 
支出に占める食費の割合が一番高い住民の名前とパーセンテージを教えて

--- SQL ---
SELECT r."NAME" AS "名前",
       (SUM(CASE WHEN UPPER(s3."KIND")=UPPER('食費') THEN s3."VALUE" ELSE 0 END)/s2."VALUE"*100) AS "パーセンテージ"
FROM "DEMO"."RESIDENTS" r
JOIN "DEMO"."RESIDENTS_SUB2" s2 ON r."ID"=s2."ID"
JOIN "DEMO"."RESIDENTS_SUB3" s3 ON s2."ID"=s3."SUB_ID"
GROUP BY r."NAME",s2."VALUE"
ORDER BY "パーセンテージ" DESC
FETCH FIRST 1 ROWS ONLY
--- RESULT ---


名前,パーセンテージ
John,33.333333


#### 以下はユーティリティ

In [20]:
# プロファイルをリスト
if not select_ai.is_connected():
    select_ai.connect(user=user, password=password, dsn=dsn)
profile = select_ai.Profile()
for fetched_profile in profile.list():
    print(fetched_profile.profile_name)

DEMO_PROFILE_PLAIN
DEMO_PROFILE_GRAPH


In [21]:
# プロファイルの削除
#if not select_ai.is_connected():
#    select_ai.connect(user=user, password=password, dsn=dsn)
#select_ai.Profile().fetch("SALES_NL2SQL").delete()